# S4 · AndinaLog 03B · Notebook 2 · Tratamiento controlado de telemetría IoT

La salida `andinalog_iot_telemetry_silver.csv` contiene solo lecturas utilizables: temperatura convertida a Celsius, unidad C y código de camión preparado. El archivo tratado completo conserva originales, decisiones y cuarentena para auditoría.

Este notebook trabaja únicamente con `andinalog_iot_telemetry.csv`. Lee las salidas del notebook 1 y el `catalogo_reglas_tratamiento.csv` de esta carpeta. Solo aplica una regla cuando el catálogo indica `APROBADA`. Conserva todas las filas originales y documenta los tratamientos aplicados y los problemas que permanecen pendientes. No fuerza ninguna fila a salir de cuarentena.

El catálogo es una decisión del proyecto, no una inferencia automática. Antes de cambiar una regla de `PENDIENTE` a `APROBADA`, documenta su evidencia y validación. El informe final se genera a partir de lo que **realmente ocurrió** en la ejecución.


## 1 · Configuración y entradas

En local, ejecuta desde cualquier carpeta del proyecto. En Colab, selecciona `drive` y ajusta la ruta de la carpeta que contiene `datasets/` y `S4/`. El notebook 2 lee los dos CSV del notebook 1 y la referencia de flota para verificar `camion_id`; guarda las salidas en `S4/andinalog_iot_telemetry/notebook2/salidas/`.


In [2]:
from pathlib import Path
import hashlib
import os
import tempfile
import pandas as pd
import numpy as np

# ============================================================
# CONFIGURACIÓN
# ============================================================

# "auto"  -> detecta automáticamente Colab o entorno local
# "local" -> fuerza ejecución local
# "drive" -> fuerza ejecución en Google Colab + Drive
ENTORNO = "auto"

RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
VERSION_TRATAMIENTO = "GIAD-M3-S4-IOT-tratamiento-v2"


# ============================================================
# DETECCIÓN DE LA RAÍZ DEL PROYECTO EN LOCAL
# ============================================================

def raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (
            (carpeta / "datasets" / "AndinaLog_03B_Bronce").is_dir()
            and (carpeta / "S4").is_dir()
        ):
            return carpeta

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Ejecuta el notebook dentro de practicasNotebookColab."
    )


# ============================================================
# CONFIGURACIÓN AUTOMÁTICA DE RUTAS
# ============================================================

def configurar_rutas(entorno, ruta_drive):

    # Detectar automáticamente el entorno
    if entorno == "auto":
        entorno = (
            "drive"
            if "google.colab" in __import__("sys").modules
            else "local"
        )

    # Google Colab + Google Drive
    if entorno == "drive":
        from google.colab import drive

        drive.mount("/content/drive")
        raiz = Path(ruta_drive)

    # Ejecución local
    elif entorno == "local":
        raiz = raiz_local()

    else:
        raise ValueError(
            "ENTORNO debe ser 'auto', 'local' o 'drive'"
        )

    print(f"Entorno detectado: {entorno}")
    print(f"Raíz del proyecto: {raiz}")

    caso = raiz / "S4" / "andinalog_iot_telemetry"

    return {
        "bronze":
            raiz
            / "datasets"
            / "AndinaLog_03B_Bronce"
            / "andinalog_iot_telemetry.csv",

        "flota":
            raiz
            / "datasets"
            / "AndinaLog_03B_Bronce"
            / "andinalog_flota.csv",

        "principal":
            caso
            / "notebook1"
            / "salidas"
            / "andinalog_iot_telemetry_diagnosticado.csv",

        "problemas":
            caso
            / "notebook1"
            / "salidas"
            / "andinalog_iot_telemetry_problemas.csv",

        "reporte1":
            caso
            / "notebook1"
            / "salidas"
            / "andinalog_iot_telemetry_reporte_calidad.csv",

        "catalogo":
            caso
            / "notebook2"
            / "catalogo_reglas_tratamiento.csv",

        "salidas":
            caso
            / "notebook2"
            / "salidas",
    }


# ============================================================
# OBTENER RUTAS
# ============================================================

rutas = configurar_rutas(
    ENTORNO,
    RUTA_PROYECTO_DRIVE
)


# ============================================================
# VERIFICAR ARCHIVOS DE ENTRADA
# ============================================================

for nombre in [
    "bronze",
    "flota",
    "principal",
    "problemas",
    "reporte1",
    "catalogo",
]:
    if not rutas[nombre].is_file():
        raise FileNotFoundError(
            f"Falta {nombre}: {rutas[nombre]}"
        )


print("Entradas verificadas.")
print("Directorio de salidas:", rutas["salidas"])


Entorno detectado: local
Raíz del proyecto: c:\Users\remrodri\Github\practicasNotebookColab
Entradas verificadas.
Directorio de salidas: c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_iot_telemetry\notebook2\salidas


## 2 · Lectura y validación del contrato

El número `fila_bronze` vincula el archivo principal con el detalle de problemas. La huella del reporte 1 debe corresponder al CSV Bronze actual; si el origen cambió, se debe volver a ejecutar el notebook 1 antes de curar.


In [3]:
def leer_entradas(rutas):
    principal = pd.read_csv(rutas["principal"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    problemas = pd.read_csv(rutas["problemas"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    reporte1 = pd.read_csv(rutas["reporte1"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    catalogo = pd.read_csv(rutas["catalogo"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    flota = pd.read_csv(rutas["flota"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    return principal, problemas, reporte1, catalogo, flota

def validar_entradas(principal, problemas, reporte1, catalogo, flota, ruta_bronze):
    requeridas = {"fila_bronze", "timestamp", "viaje_id", "temperatura_cabina_c", "temp_unit", "humedad_cabina_pct", "en_cuarentena"}
    if not requeridas.issubset(principal.columns):
        raise ValueError(f"Faltan columnas del principal: {sorted(requeridas - set(principal.columns))}")
    if not {"fila_bronze", "columna_afectada", "codigo_error", "valor_original"}.issubset(problemas.columns):
        raise ValueError("El detalle de problemas no cumple su contrato")
    if not {"regla_id", "estado", "evidencia_acuerdo"}.issubset(catalogo.columns):
        raise ValueError("El catálogo no cumple su contrato")
    if principal["fila_bronze"].duplicated().any() or catalogo["regla_id"].duplicated().any():
        raise ValueError("Hay identificadores duplicados")
    if not problemas["fila_bronze"].isin(principal["fila_bronze"]).all():
        raise ValueError("Hay problemas sin fila en el principal")
    if not catalogo["estado"].isin(["APROBADA", "PENDIENTE"]).all():
        raise ValueError("Estado de regla desconocido")
    if "camion_id" not in flota.columns:
        raise ValueError("La referencia de flota no contiene camion_id")
    flota_unica = flota.drop_duplicates().copy()
    if flota_unica["camion_id"].str.strip().str.upper().duplicated().any():
        raise ValueError("Un mismo camion_id tiene datos contradictorios en flota")
    hash_reporte = reporte1.set_index("metrica").loc["sha256_bronze", "valor"]
    hash_actual = hashlib.sha256(ruta_bronze.read_bytes()).hexdigest()
    if hash_reporte != hash_actual:
        raise ValueError("El Bronze ya no coincide con el diagnóstico; ejecuta el notebook 1")
    if len(principal) != int(reporte1.set_index("metrica").loc["filas_bronze", "valor"]):
        raise ValueError("El principal no coincide con el conteo Bronze")
    return hash_actual

df_entrada, problemas_entrada, reporte1, catalogo, flota = leer_entradas(rutas)
HASH_BRONZE = validar_entradas(df_entrada, problemas_entrada, reporte1, catalogo, flota, rutas["bronze"])
HASH_FLOTA = hashlib.sha256(rutas["flota"].read_bytes()).hexdigest()
print(f"Filas: {len(df_entrada):,}; problemas iniciales: {len(problemas_entrada):,}")
display(catalogo)


Filas: 28,920; problemas iniciales: 455


,regla_id,columna_afectada,codigo_error_o_unidad,tratamiento_propuesto,estado,evidencia_acuerdo,validacion_requerida
0,TEMP_C_CONSERVAR,temperatura_cabina_c,C,Conservar el valor en Celsius en columna prepa...,APROBADA,C ya representa Celsius; conservar el valor or...,Valor numérico y unidad C
1,TEMP_F_A_C,temperatura_cabina_c,F,Convertir F a C con (F-32)*5/9,APROBADA,Conversión de F a Celsius confirmada por el us...,Valor numérico y unidad F
2,TEMP_K_A_C,temp_unit,UNIDAD_NO_RECONOCIDA,Convertir K a C con K-273.15,APROBADA,K confirmado como kelvin por el usuario el 202...,K confirmado como kelvin; valor numérico no ne...
3,DUPLICADO_IDENTICO,viaje_id+timestamp,DUPLICADO,Elegir primera lectura canónica y excluir copi...,APROBADA,Regla de duplicados aprobada por el usuario el...,Misma clave y diez columnas originales idénticas
4,DUPLICADO_MAYUSCULAS,viaje_id+timestamp,DUPLICADO,Elegir primera lectura y excluir copia con cam...,APROBADA,Regla aprobada por el usuario; CAM-12 existe e...,Solo cambia mayúscula/minúscula de camion_id; ...
5,DUPLICADO_COMPLEMENTARIO,viaje_id+timestamp,DUPLICADO,Elegir lectura más completa y excluir la incom...,APROBADA,Regla de duplicados aprobada por el usuario el...,Todos los valores no vacíos coinciden y una le...
6,FALTANTE_TEMPERATURA,temperatura_cabina_c,FALTANTE,Sin imputación automática,PENDIENTE,,Fuente recuperable o método aprobado
7,FALTANTE_HUMEDAD,humedad_cabina_pct,FALTANTE,Sin imputación automática,PENDIENTE,,Fuente recuperable o método aprobado
8,HUMEDAD_FUERA_RANGO,humedad_cabina_pct,FUERA_RANGO,Conservar en cuarentena,PENDIENTE,,Lectura original verificable
9,FECHA_INVALIDA,timestamp,FECHA_INVALIDA,Conservar en cuarentena,PENDIENTE,,Timestamp real recuperado de fuente autorizada


## 3 · Aplicación de reglas aprobadas

El valor original no se sobrescribe. La conversión C/F se guarda en `temperatura_c_preparada` cuando su regla está aprobada. Kelvin se convierte solo si la aprobación confirma explícitamente que `K` es kelvin. Fechas imposibles, humedades negativas y faltantes no se corrigen por conjetura. Los duplicados se comparan antes de elegir una canónica: copia idéntica, diferencia de mayúsculas confirmada en flota o lectura más completa sin valores no vacíos contradictorios. La copia queda en el archivo completo y en cuarentena.


In [4]:
def regla_aprobada(catalogo, regla_id):
    fila = catalogo.loc[catalogo["regla_id"].eq(regla_id)]
    if len(fila) != 1:
        raise ValueError(f"Falta regla única: {regla_id}")
    aprobada = fila.iloc[0]["estado"] == "APROBADA"
    if aprobada and not str(fila.iloc[0]["evidencia_acuerdo"]).strip():
        raise ValueError(f"La regla {regla_id} figura aprobada sin evidencia del acuerdo")
    return aprobada

def preparar_temperatura(principal, catalogo):
    df = principal.copy(deep=True)
    numero = pd.to_numeric(df["temperatura_cabina_c"].str.strip(), errors="coerce")
    unidad = df["temp_unit"].str.strip().str.upper()
    df["temperatura_c_preparada"] = np.nan
    df["tratamiento_temperatura"] = "SIN_TRATAMIENTO"
    if regla_aprobada(catalogo, "TEMP_C_CONSERVAR"):
        c = unidad.eq("C") & numero.notna()
        df.loc[c, "temperatura_c_preparada"] = numero[c]
        df.loc[c, "tratamiento_temperatura"] = "C_ORIGINAL"
    if regla_aprobada(catalogo, "TEMP_F_A_C"):
        f = unidad.eq("F") & numero.notna()
        df.loc[f, "temperatura_c_preparada"] = (numero[f] - 32) * 5 / 9
        df.loc[f, "tratamiento_temperatura"] = "F_A_C"
    if regla_aprobada(catalogo, "TEMP_K_A_C"):
        k = unidad.eq("K") & numero.notna() & numero.ge(0)
        df.loc[k, "temperatura_c_preparada"] = numero[k] - 273.15
        df.loc[k, "tratamiento_temperatura"] = "K_A_C"
    df["temperatura_c_preparada"] = df["temperatura_c_preparada"].round(6)
    return df

def analizar_duplicados(principal, catalogo, flota):
    columnas_raw = ["timestamp", "viaje_id", "order_id", "camion_id", "producto_id", "temperatura_cabina_c", "temp_unit", "humedad_cabina_pct", "desviacion_termica_flag", "desviacion_proximos_60min_flag"]
    ids_flota = set(flota["camion_id"].str.strip().str.upper())
    trabajo = principal.copy()
    trabajo["_clave"] = trabajo["viaje_id"].str.strip() + "|" + trabajo["timestamp"].str.strip()
    registros = []
    aprobaciones = {
        "IDENTICO": regla_aprobada(catalogo, "DUPLICADO_IDENTICO"),
        "MAYUSCULAS": regla_aprobada(catalogo, "DUPLICADO_MAYUSCULAS"),
        "COMPLEMENTARIO": regla_aprobada(catalogo, "DUPLICADO_COMPLEMENTARIO"),
    }
    for clave, grupo in trabajo.groupby("_clave", sort=False):
        if len(grupo) == 1:
            continue
        grupo = grupo.sort_values("fila_bronze", key=lambda s: s.astype(int))
        tipo, canonica, evidencia = "CONFLICTO_O_AMBIGUO", "", "No se eligió lectura canónica sin evidencia suficiente"
        if len(grupo) == 2:
            primera, segunda = grupo.iloc[0], grupo.iloc[1]
            exacto = all(primera[c] == segunda[c] for c in columnas_raw)
            camion_equivalente = (primera["camion_id"].strip().upper() == segunda["camion_id"].strip().upper()
                                   and primera["camion_id"].strip().upper() in ids_flota)
            solo_mayusculas = (not exacto and camion_equivalente and primera["camion_id"] != segunda["camion_id"]
                               and all(primera[c] == segunda[c] for c in columnas_raw if c != "camion_id"))
            def comparable(fila, columna):
                valor = fila[columna].strip()
                return valor.upper() if columna == "camion_id" and valor.upper() in ids_flota else valor
            sin_conflicto = all(not (comparable(primera, c) and comparable(segunda, c)
                                    and comparable(primera, c) != comparable(segunda, c)) for c in columnas_raw)
            n_primera = sum(bool(primera[c].strip()) for c in columnas_raw)
            n_segunda = sum(bool(segunda[c].strip()) for c in columnas_raw)
            if exacto:
                tipo, canonica, evidencia = "IDENTICO", primera["fila_bronze"], "Las diez columnas originales son idénticas"
            elif solo_mayusculas:
                tipo, canonica, evidencia = "MAYUSCULAS", primera["fila_bronze"], "Solo cambia la escritura de camion_id; el código normalizado existe en flota"
            elif sin_conflicto and n_primera != n_segunda:
                canonica = primera["fila_bronze"] if n_primera > n_segunda else segunda["fila_bronze"]
                tipo, evidencia = "COMPLEMENTARIO", "Una lectura tiene más campos completos y todos los valores no vacíos coinciden"
        autorizada = aprobaciones.get(tipo, False)
        for _, fila in grupo.iterrows():
            decision = ("CANONICA" if fila["fila_bronze"] == canonica else "COPIA_EXCLUIDA") if autorizada else "PENDIENTE"
            registros.append({"fila_bronze": fila["fila_bronze"], "clave_lectura": clave,
                              "tipo_duplicado": tipo, "decision_duplicado": decision,
                              "fila_canonica": canonica if autorizada else "",
                              "justificacion_duplicado": evidencia if autorizada else "Regla no aprobada o evidencia insuficiente",
                              "camion_id_preparado": fila["camion_id"].strip().upper() if tipo == "MAYUSCULAS" and autorizada else fila["camion_id"]})
    return pd.DataFrame(registros)

def evaluar_problemas(principal, problemas, catalogo, decisiones):
    acciones = problemas.copy(deep=True)
    acciones["estado_tratamiento"] = "PENDIENTE"
    acciones["tratamiento_aplicado"] = "NINGUNO"
    acciones["detalle_resultado"] = "No hay regla aprobada y validada para liberar esta fila"
    if regla_aprobada(catalogo, "TEMP_K_A_C"):
        k = acciones["columna_afectada"].eq("temp_unit") & acciones["codigo_error"].eq("UNIDAD_NO_RECONOCIDA") & acciones["valor_original"].str.strip().str.upper().eq("K")
        temperatura = principal.set_index("fila_bronze")["temperatura_c_preparada"]
        resuelta = k & acciones["fila_bronze"].map(temperatura).notna()
        acciones.loc[resuelta, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = ["RESUELTO", "K_A_C", "Unidad confirmada y conversión validada; revisar otros problemas de la fila"]
    por_fila = decisiones.set_index("fila_bronze")
    dup = acciones["codigo_error"].eq("DUPLICADO")
    decision = acciones["fila_bronze"].map(por_fila["decision_duplicado"])
    elegida = dup & decision.eq("CANONICA")
    excluida = dup & decision.eq("COPIA_EXCLUIDA")
    acciones.loc[elegida, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = ["RESUELTO", "SELECCION_CANONICA", "Lectura canónica elegida por integridad y coincidencia de valores; copia conservada"]
    acciones.loc[excluida, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = ["EXCLUIDO_COMO_COPIA", "COPIA_EXCLUIDA", "Copia no canónica conservada para auditoría; no utilizable como nueva lectura"]
    # Si la primera lectura incompleta fue sustituida por la segunda, la primera
    # no tenía una bandera DUPLICADO inicial: se registra la decisión adicional.
    existentes = set(acciones.loc[dup, "fila_bronze"])
    nuevas = decisiones.loc[decisiones["decision_duplicado"].eq("COPIA_EXCLUIDA")
                            & ~decisiones["fila_bronze"].isin(existentes)]
    if len(nuevas):
        extra = pd.DataFrame({"fila_bronze": nuevas["fila_bronze"],
                              "columna_afectada": "viaje_id+timestamp", "codigo_error": "DUPLICADO",
                              "valor_original": nuevas["clave_lectura"],
                              "version_diagnostico": problemas["version_diagnostico"].iloc[0],
                              "estado_tratamiento": "EXCLUIDO_COMO_COPIA",
                              "tratamiento_aplicado": "COPIA_EXCLUIDA",
                              "detalle_resultado": "Primera lectura incompleta sustituida por lectura canónica más completa"})
        acciones = pd.concat([acciones, extra], ignore_index=True)
    acciones = acciones.sort_values(["fila_bronze", "columna_afectada"], key=lambda s: s.astype(int) if s.name == "fila_bronze" else s, kind="stable")
    return acciones

df_trabajo = preparar_temperatura(df_entrada, catalogo)
decisiones_duplicado = analizar_duplicados(df_trabajo, catalogo, flota)
acciones = evaluar_problemas(df_trabajo, problemas_entrada, catalogo, decisiones_duplicado)
print(decisiones_duplicado.groupby(["tipo_duplicado", "decision_duplicado"]).size().to_string())
print(acciones["estado_tratamiento"].value_counts().to_string())


tipo_duplicado  decision_duplicado
COMPLEMENTARIO  CANONICA                5
                COPIA_EXCLUIDA          5
IDENTICO        CANONICA              114
                COPIA_EXCLUIDA        114
MAYUSCULAS      CANONICA                1
                COPIA_EXCLUIDA          1
estado_tratamiento
PENDIENTE              330
EXCLUIDO_COMO_COPIA    120
RESUELTO                 6


## 4 · Estado final y conciliación

Solo `RESUELTO` deja de ser motivo pendiente. `EXCLUIDO_COMO_COPIA` conserva la fila en cuarentena para impedir que duplique una lectura, aunque su tratamiento queda registrado. Una fila sale de cuarentena únicamente cuando no le queda ningún problema pendiente o excluido.


In [5]:
def construir_estado_final(principal, acciones, decisiones):
    df = principal.copy(deep=True)
    por_fila = decisiones.set_index("fila_bronze")
    for campo in ["tipo_duplicado", "decision_duplicado", "fila_canonica", "justificacion_duplicado"]:
        df[campo] = df["fila_bronze"].map(por_fila[campo]).fillna("")
    df["camion_id_preparado"] = df["fila_bronze"].map(por_fila["camion_id_preparado"]).fillna(df["camion_id"])
    df["en_cuarentena_inicial"] = df["en_cuarentena"].str.lower().eq("true")
    df["columnas_problema_iniciales"] = df["columnas_con_problemas"]
    pendientes = acciones.loc[~acciones["estado_tratamiento"].eq("RESUELTO")].copy()
    pendientes["motivo"] = pendientes["columna_afectada"] + ":" + pendientes["codigo_error"]
    motivo_por_fila = pendientes.groupby("fila_bronze")["motivo"].agg(lambda v: "|".join(v))
    df["motivos_finales"] = df["fila_bronze"].map(motivo_por_fila).fillna("")
    df["en_cuarentena_final"] = df["motivos_finales"].ne("")
    tratamientos = acciones.loc[acciones["tratamiento_aplicado"].ne("NINGUNO")].groupby("fila_bronze")["tratamiento_aplicado"].agg(lambda v: "|".join(dict.fromkeys(v)))
    tratamiento_medicion = df["tratamiento_temperatura"].where(df["tratamiento_temperatura"].isin(["F_A_C", "K_A_C"]), "")
    df["tratamientos_aplicados"] = ["|".join(dict.fromkeys([x for x in [t, a] if x]))
                                   for t, a in zip(tratamiento_medicion, df["fila_bronze"].map(tratamientos).fillna(""))]
    df["version_tratamiento"] = VERSION_TRATAMIENTO
    return df, pendientes

df_final, problemas_pendientes = construir_estado_final(df_trabajo, acciones, decisiones_duplicado)
df_cuarentena_final = df_final.loc[df_final["en_cuarentena_final"]].copy()
# Silver de consumo: solo lecturas utilizables, con temperatura y camión preparados.
df_silver = df_final.loc[~df_final["en_cuarentena_final"], [
    "timestamp", "viaje_id", "order_id", "camion_id_preparado", "producto_id",
    "temperatura_c_preparada", "humedad_cabina_pct",
    "desviacion_termica_flag", "desviacion_proximos_60min_flag",
]].copy()
df_silver = df_silver.rename(columns={
    "camion_id_preparado": "camion_id", "temperatura_c_preparada": "temperatura_cabina_c",
})
df_silver.insert(6, "temp_unit", "C")

pd.testing.assert_frame_equal(df_final[df_entrada.columns], df_entrada)
assert len(df_final) == len(df_entrada)
assert len(df_cuarentena_final) == int(df_final["en_cuarentena_final"].sum())
assert len(df_silver) == int((~df_final["en_cuarentena_final"]).sum())
assert list(df_silver.columns) == ["timestamp", "viaje_id", "order_id", "camion_id", "producto_id", "temperatura_cabina_c", "temp_unit", "humedad_cabina_pct", "desviacion_termica_flag", "desviacion_proximos_60min_flag"]
assert df_silver["temperatura_cabina_c"].notna().all()
assert df_silver["temp_unit"].eq("C").all()
assert not df_silver.duplicated(["viaje_id", "timestamp"]).any()
assert set(problemas_pendientes["fila_bronze"]) == set(df_cuarentena_final["fila_bronze"])
assert df_final.loc[df_final["decision_duplicado"].eq("COPIA_EXCLUIDA"), "en_cuarentena_final"].all()
assert not df_final.loc[~df_final["en_cuarentena_final"]].duplicated(["viaje_id", "timestamp"]).any()
selecciones = decisiones_duplicado.groupby("clave_lectura")["decision_duplicado"].value_counts().unstack(fill_value=0)
aprobadas = selecciones.loc[selecciones.get("PENDIENTE", pd.Series(0, index=selecciones.index)).eq(0)]
assert aprobadas.get("CANONICA", pd.Series(0, index=aprobadas.index)).eq(1).all()
assert not (df_final["en_cuarentena_inicial"] & ~df_final["en_cuarentena_final"]).any() or (acciones["estado_tratamiento"].eq("RESUELTO")).any()
print("Filas iniciales en cuarentena:", int(df_final["en_cuarentena_inicial"].sum()))
print("Filas finales en cuarentena:", int(df_final["en_cuarentena_final"].sum()))
print("Filas liberadas:", int((df_final["en_cuarentena_inicial"] & ~df_final["en_cuarentena_final"]).sum()))


Filas iniciales en cuarentena: 450
Filas finales en cuarentena: 445
Filas liberadas: 5


## 5 · Informe de lo aplicado y exportación

El reporte registra las reglas aprobadas, los problemas resueltos, los pendientes, las copias excluidas y el número real de filas liberadas. No atribuye una liberación a una conversión si la fila mantiene otro motivo.


In [6]:
def crear_reporte(df_final, acciones, decisiones, catalogo, huella, huella_flota):
    base = [
        ("sha256_bronze", huella),
        ("sha256_flota", huella_flota),
        ("version_tratamiento", VERSION_TRATAMIENTO),
        ("filas_totales", len(df_final)),
        ("filas_cuarentena_inicial", int(df_final["en_cuarentena_inicial"].sum())),
        ("filas_cuarentena_final", int(df_final["en_cuarentena_final"].sum())),
        ("filas_liberadas", int((df_final["en_cuarentena_inicial"] & ~df_final["en_cuarentena_final"]).sum())),
        ("problemas_resueltos", int(acciones["estado_tratamiento"].eq("RESUELTO").sum())),
        ("copias_excluidas", int(acciones["estado_tratamiento"].eq("EXCLUIDO_COMO_COPIA").sum())),
        ("problemas_pendientes", int(acciones["estado_tratamiento"].eq("PENDIENTE").sum())),
        ("temperaturas_C_conservadas", int(df_final["tratamiento_temperatura"].eq("C_ORIGINAL").sum())),
        ("temperaturas_F_a_C", int(df_final["tratamiento_temperatura"].eq("F_A_C").sum())),
        ("temperaturas_K_a_C", int(df_final["tratamiento_temperatura"].eq("K_A_C").sum())),
    ]
    canonicas = decisiones.loc[decisiones["decision_duplicado"].eq("CANONICA")]
    base += [("pares_duplicados_" + tipo.lower(), int(canonicas["tipo_duplicado"].eq(tipo).sum()))
             for tipo in ["IDENTICO", "MAYUSCULAS", "COMPLEMENTARIO"]]
    base += [("duplicados_sin_decision", int(decisiones["decision_duplicado"].eq("PENDIENTE").sum()))]
    base += [("regla_" + r["regla_id"], r["estado"]) for _, r in catalogo.iterrows()]
    return pd.DataFrame(base, columns=["metrica", "valor"])

def exportar(directorio, tablas, bronze, huella, flota, huella_flota):
    if hashlib.sha256(bronze.read_bytes()).hexdigest() != huella:
        raise RuntimeError("El Bronze cambió; se cancela la exportación")
    if hashlib.sha256(flota.read_bytes()).hexdigest() != huella_flota:
        raise RuntimeError("La referencia de flota cambió; se cancela la exportación")
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_iot2_", dir=directorio, encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

reporte2 = crear_reporte(df_final, acciones, decisiones_duplicado, catalogo, HASH_BRONZE, HASH_FLOTA)
tablas = {
    "andinalog_iot_telemetry_tratado.csv": df_final,
    "andinalog_iot_telemetry_silver.csv": df_silver,
    "andinalog_iot_telemetry_acciones.csv": acciones,
    "andinalog_iot_telemetry_decisiones_duplicados.csv": decisiones_duplicado,
    "andinalog_iot_telemetry_cuarentena_final.csv": df_cuarentena_final,
    "andinalog_iot_telemetry_reporte_tratamiento.csv": reporte2,
}
for ruta in exportar(rutas["salidas"], tablas, rutas["bronze"], HASH_BRONZE, rutas["flota"], HASH_FLOTA):
    print(ruta)
display(reporte2)


c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_iot_telemetry\notebook2\salidas\andinalog_iot_telemetry_tratado.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_iot_telemetry\notebook2\salidas\andinalog_iot_telemetry_silver.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_iot_telemetry\notebook2\salidas\andinalog_iot_telemetry_acciones.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_iot_telemetry\notebook2\salidas\andinalog_iot_telemetry_decisiones_duplicados.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_iot_telemetry\notebook2\salidas\andinalog_iot_telemetry_cuarentena_final.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_iot_telemetry\notebook2\salidas\andinalog_iot_telemetry_reporte_tratamiento.csv


,metrica,valor
0,sha256_bronze,c5f78ed802f8455dfc8eac91295954c38a159029e77b8e...
1,sha256_flota,5c02eb8ef591c7f98846b8f547f41383467b41e7768bdd...
2,version_tratamiento,GIAD-M3-S4-IOT-tratamiento-v2
3,filas_totales,28920
4,filas_cuarentena_inicial,450
5,filas_cuarentena_final,445
6,filas_liberadas,5
7,problemas_resueltos,6
8,copias_excluidas,120
9,problemas_pendientes,330


In [7]:
def crear_informe_md(df_final, acciones, decisiones, catalogo, huella, huella_flota):
    inicial = int(df_final["en_cuarentena_inicial"].sum())
    final = int(df_final["en_cuarentena_final"].sum())
    liberadas = inicial - final
    f = int(df_final["tratamiento_temperatura"].eq("F_A_C").sum())
    k = int(df_final["tratamiento_temperatura"].eq("K_A_C").sum())
    resueltos = int(acciones["estado_tratamiento"].eq("RESUELTO").sum())
    pendientes = int(acciones["estado_tratamiento"].eq("PENDIENTE").sum())
    excluidos = int(acciones["estado_tratamiento"].eq("EXCLUIDO_COMO_COPIA").sum())
    canonicas = decisiones.loc[decisiones["decision_duplicado"].eq("CANONICA")]
    exactos = int(canonicas["tipo_duplicado"].eq("IDENTICO").sum())
    mayusculas = int(canonicas["tipo_duplicado"].eq("MAYUSCULAS").sum())
    complementarios = int(canonicas["tipo_duplicado"].eq("COMPLEMENTARIO").sum())
    liberadas_k = int((df_final["temp_unit"].str.strip().str.upper().eq("K") & df_final["en_cuarentena_inicial"] & ~df_final["en_cuarentena_final"]).sum())
    liberadas_duplicado = int((df_final["decision_duplicado"].eq("CANONICA") & df_final["en_cuarentena_inicial"] & ~df_final["en_cuarentena_final"]).sum())
    frase_duplicado = ("Una lectura canónica antes marcada como duplicada salió de cuarentena."
                       if liberadas_duplicado == 1 else
                       f"{liberadas_duplicado:,} lecturas canónicas antes marcadas como duplicadas salieron de cuarentena.")
    conteos = acciones.loc[~acciones["estado_tratamiento"].eq("RESUELTO")].groupby(["columna_afectada", "codigo_error"]).size()
    lineas = [
        "# Informe de tratamiento de telemetría IoT",
        "",
        f"**Fuente Bronze SHA-256:** `{huella}`",
        f"**Referencia de flota SHA-256:** `{huella_flota}`",
        f"**Versión:** `{VERSION_TRATAMIENTO}`",
        "",
        "## Resultado del lote",
        "",
        f"Se conservaron las {len(df_final):,} filas. La cuarentena pasó de {inicial:,} a {final:,} filas; {liberadas:,} salieron después de resolver todos sus motivos. Se registraron {resueltos:,} problemas resueltos, {pendientes:,} pendientes y {excluidos:,} copias excluidas de la vista utilizable.",
        "",
        "## Tratamientos aplicados",
        "",
        f"- Se convirtieron {f:,} temperaturas de Fahrenheit a Celsius con `(F − 32) × 5/9`, conservando el valor original. Estas lecturas no estaban en cuarentena únicamente por la unidad.",
        f"- Se convirtieron {k:,} temperaturas de kelvin a Celsius con `K − 273,15`, según la confirmación del usuario. {liberadas_k:,} filas salieron de cuarentena; las demás conservaron un motivo adicional.",
        "- No se imputaron temperaturas ni humedades. No se corrigieron fechas imposibles ni humedades negativas por suposición.",
        "",
        "## Duplicados y selección canónica",
        "",
        f"Se revisaron {exactos + mayusculas + complementarios:,} pares de lecturas con la misma clave `viaje_id + timestamp`: {exactos:,} pares idénticos, {mayusculas:,} con una diferencia de mayúsculas en `camion_id` confirmada por la referencia de flota y {complementarios:,} con una lectura más completa sin valores no vacíos contradictorios.",
        f"Se eligió una lectura canónica por par y se conservaron {excluidos:,} copias como `EXCLUIDO_COMO_COPIA` en el archivo completo y en cuarentena. {frase_duplicado} Una lectura incompleta anterior recibió una acción adicional de copia sustituida. Ninguna copia se borró ni se contó como una nueva medición utilizable.",
        "Cualquier par con valores no vacíos contradictorios, sin una lectura claramente más completa o sin un código de camión verificable permanece pendiente. La elección de cada par y su justificación constan en `andinalog_iot_telemetry_decisiones_duplicados.csv`.",
        "",
        "## Motivos que permanecen en cuarentena",
        "",
        "| Columna | Código | Motivos finales |",
        "|---|---|---:|",
    ]
    lineas += [f"| {col} | {codigo} | {int(total)} |" for (col, codigo), total in conteos.items()]
    lineas += [
        "",
        "La cantidad de motivos finales puede superar las filas en cuarentena porque una fila puede presentar más de un problema. Los duplicados excluidos figuran como motivos finales para impedir su uso como nuevas lecturas. Cada intento y su resultado constan en el CSV de acciones; el CSV tratado mantiene los valores originales y el estado inicial y final.",
        "",
        "## Reglas y límites de esta ejecución",
        "",
    ]
    lineas += [f"- `{r['regla_id']}`: **{r['estado']}**. Tratamiento: {r['tratamiento_propuesto']}. Validación: {r['validacion_requerida']}. Acuerdo: {r['evidencia_acuerdo'] or 'pendiente.'}"
               for _, r in catalogo.iterrows()]
    lineas += ["", "Una fila permaneció en cuarentena cuando no había una regla aprobada, faltaba evidencia o quedaba otro motivo sin resolver. No se forzó ninguna liberación.", ""]
    return "\n".join(lineas)

informe_md = crear_informe_md(df_final, acciones, decisiones_duplicado, catalogo, HASH_BRONZE, HASH_FLOTA)
ruta_informe = rutas["salidas"] / "Informe_S4_02_Tratamiento.md"
ruta_informe.write_text(informe_md, encoding="utf-8")
print(ruta_informe)


c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_iot_telemetry\notebook2\salidas\Informe_S4_02_Tratamiento.md


## 6 · Interpretación de esta ejecución

Revisa el reporte y el archivo de acciones antes de afirmar que una fila salió de cuarentena. Las reglas que siguen `PENDIENTE` requieren evidencia o acuerdo; por eso no se aplica una imputación, una fecha supuesta ni un valor de humedad fabricado. El informe de curación debe describir exactamente los intentos ejecutados, sus resultados y los casos que permanecen en cuarentena.
